In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu130
True


In [2]:
!nvidia-smi
!git clone https://github.com/mk-minchul/AdaFace.git
%cd AdaFace
!ls

Thu Oct 22 12:21:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!head -100 README.md

# AdaFace: Quality Adaptive Margin for Face Recognition

Official github repository for AdaFace: Quality Adaptive Margin for Face Recognition. 
The paper (https://arxiv.org/abs/2204.00964) is presented in CVPR 2022 (Oral). 


> Abstract: Recognition in low quality face datasets is challenging because facial attributes are obscured and degraded. Advances in margin-based loss functions have resulted in enhanced discriminability of faces in the embedding space. Further, previous studies have studied the effect of adaptive losses to assign more importance to misclassified (hard) examples. In this work, we introduce another aspect of adaptiveness in the loss function, namely the image quality. We argue that the strategy to emphasize misclassified samples should be adjusted according to their image quality. Specifically, the relative importance of easy and hard samples should be based on the sample's image quality. We propose a new loss function that emphasizes samples of different difficult

In [4]:
!tail -n +100 README.md
!cat requirements.txt


| Arch | Dataset    | Link                                                                                         |
|------|------------|----------------------------------------------------------------------------------------------|
| R18  | CASIA-WebFace     | [gdrive](https://drive.google.com/file/d/1BURBDplf2bXpmwOL1WVzqtaVmQl9NpPe/view?usp=sharing) |
| R18  | VGGFace2     | [gdrive](https://drive.google.com/file/d/1k7onoJusC0xjqfjB-hNNaxz9u6eEzFdv/view?usp=sharing) |
| R18  | WebFace4M     | [gdrive](https://drive.google.com/file/d/1J17_QW1Oq00EhSWObISnhWEYr2NNrg2y/view?usp=sharing) |
| R50  | CASIA-WebFace     | [gdrive](https://drive.google.com/file/d/1g1qdg7_HSzkue7_VrW64fnWuHl0YL2C2/view?usp=sharing) |
| R50  | WebFace4M     | [gdrive](https://drive.google.com/file/d/1BmDRrhPsHSbXcWZoYFPJg2KJn1sd3QpN/view?usp=sharing) |
| R50  | MS1MV2     | [gdrive](https://drive.google.com/file/d/1eUaSHG4pGlIZK7hBkqjyp2fc2epKoBvI/view?usp=sharing) |
| R100 | MS1MV2     | [gdrive](https://dr

In [5]:
!pip install -q gdown
!mkdir -p pretrained
!gdown --fuzzy "https://drive.google.com/file/d/1BURBDplf2bXpmwOL1WVzqtaVmQl9NpPe/view?usp=sharing" -O pretrained/adaface_ir18_casia.ckpt
!ls -lh pretrained

Downloading...
From (original): https://drive.google.com/uc?id=1BURBDplf2bXpmwOL1WVzqtaVmQl9NpPe
From (redirected): https://drive.google.com/uc?id=1BURBDplf2bXpmwOL1WVzqtaVmQl9NpPe&confirm=t&uuid=4d40459f-82fe-4a35-b161-4cf6c8ba6f6e
To: /content/AdaFace/pretrained/adaface_ir18_casia.ckpt
100% 118M/118M [00:05<00:00, 23.2MB/s]
total 113M
-rw-r--r-- 1 root root 113M Mar 26  2022 adaface_ir18_casia.ckpt


In [6]:
!sed -n 1,60p inference.py

import net
import torch
import os
from face_alignment import align
import numpy as np


adaface_models = {
    'ir_50':"pretrained/adaface_ir50_ms1mv2.ckpt",
}

def load_pretrained_model(architecture='ir_50'):
    # load model and pretrained statedict
    assert architecture in adaface_models.keys()
    model = net.build_model(architecture)
    statedict = torch.load(adaface_models[architecture])['state_dict']
    model_statedict = {key[6:]:val for key, val in statedict.items() if key.startswith('model.')}
    model.load_state_dict(model_statedict)
    model.eval()
    return model

def to_input(pil_rgb_image):
    np_img = np.array(pil_rgb_image)
    brg_img = ((np_img[:,:,::-1] / 255.) - 0.5) / 0.5
    tensor = torch.tensor([brg_img.transpose(2,0,1)]).float()
    return tensor

if __name__ == '__main__':

    model = load_pretrained_model('ir_50')
    feature, norm = model(torch.randn(2,3,112,112))

    test_image_path = 'face_alignment/test_images'
    features = []
    for fname in

In [7]:
!grep -n "ir_18\|ir_50\|def build_model" net.py
!cat requirements.txt

14:def build_model(model_name='ir_50'):
17:    elif model_name == 'ir_50':
23:    elif model_name == 'ir_18':
pytorch-lightning==1.8.6
pytorch<=1.13.1
tqdm
bcolz-zipline
prettytable
menpo
mxnet
opencv-python


In [8]:
!sed -i "s#'ir_50':\"pretrained/adaface_ir50_ms1mv2.ckpt\"#'ir_18':\"pretrained/adaface_ir18_casia.ckpt\"#" inference.py
!sed -i "s#load_pretrained_model('ir_50')#load_pretrained_model('ir_18')#" inference.py
!grep -n "ir_18\|ir_50" inference.py
!python inference.py

9:    'ir_18':"pretrained/adaface_ir18_casia.ckpt",
12:def load_pretrained_model(architecture='ir_50'):
30:    model = load_pretrained_model('ir_18')
/content/AdaFace/face_alignment/mtcnn_pytorch/src/align_trans.py:287: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if align_type is 'cv2_affine':
/content/AdaFace/face_alignment/mtcnn_pytorch/src/align_trans.py:290: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  elif align_type is 'affine':
/content/AdaFace/inference.py:25: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  tensor = torch.tensor([brg_img.transpose(2,0,1)]).float()
tensor([[ 1.0000,  0.6369,  0.0048],
        [ 0.6369,  1.0000, -0.0130],
        [ 0.0048, -0.0130,  1.0000]], grad_fn=<MmBackward0>)
